In [1]:
from pathlib import Path
import os
import time
import random
import torch

import numpy as np
import pandas as pd

from training import train_single
from data_helpers import make_loader
from models_transformer import SingleOutTransformerNet
from metrics import eval_single_metrics_test

In [2]:
OUTDIR = Path("./Results")
OUTDIR.mkdir(parents=True, exist_ok=True)
MODELSDIR = Path("./trained_models")
MODELSDIR.mkdir(parents=True, exist_ok=True)

In [3]:
SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cpu")
print("Device:", DEVICE)

Device: cpu


#### Load data

In [4]:
train_data = np.load("data_processed/train_data.npz")
val_data = np.load("data_processed/val_data.npz")
test_data = np.load("data_processed/test_data.npz")

X_train = train_data["x"]
y_train = train_data["age"]

X_val = val_data["x"]
y_val = val_data["age"]

X_test = test_data["x"]
y_test = test_data["age"]

In [5]:
train_data_s = np.load("data_processed/train_data_scaled.npz")
val_data_s = np.load("data_processed/val_data_scaled.npz")
test_data_s = np.load("data_processed/test_data_scaled.npz")

X_train_s = train_data_s["x"]
y_train_s = train_data_s["age"]

X_val_s = val_data_s["x"]
y_val_s = val_data_s["age"]

X_test_s = test_data_s["x"]
y_test_s = test_data_s["age"]

In [6]:
scalers = np.load("data_processed/scalers.npz")

x_mean, x_std = scalers["x_mean"], scalers["x_std"]
y_mean, y_std = scalers["age_mean"], scalers["age_std"]

In [7]:
BATCH_SIZE = 128

train_loader = make_loader(X_train_s, y_train_s, BATCH_SIZE)
val_loader = make_loader(X_val_s, y_val_s, BATCH_SIZE)
test_loader = make_loader(X_test_s, y_test_s, BATCH_SIZE)

### Models and training

In [8]:
EPOCHS = 150
# METRICS_EVERY = int(EPOCHS/8)
METRICS_EVERY = 1
LR = 1e-3
WD = 1e-5

In [9]:
IN_DIM = X_train_s.shape[1]

EMB_DIM = 64
NHEAD = 4
NUM_LAYERS = 17
FF_DIM = 128
DROPOUT = 0.1

In [10]:
tag = "single_TR_age"

torch.manual_seed(SEED)    
model_single = SingleOutTransformerNet(IN_DIM, emb_dim=EMB_DIM, nhead=NHEAD, 
                                       num_layers=NUM_LAYERS, ff_dim=FF_DIM, 
                                       dropout=DROPOUT).to(DEVICE)    

loss = torch.nn.SmoothL1Loss()
optimizer = torch.optim.Adam(model_single.parameters(), lr=LR, weight_decay=WD)

hist_df = train_single(model_single, loss, optimizer, train_loader, val_loader, 
                       device=DEVICE, epochs=EPOCHS, metrics_every=METRICS_EVERY)

hist_df.to_csv(f"Results/train_history/train_history_{NUM_LAYERS}layers.csv", index=False)
torch.save(model_single.state_dict(), MODELSDIR / f"TR_model_{NUM_LAYERS}layers.pt")

ep=001 tr_loss=0.3087 | va_loss=0.1945 | tr R2=0.598 | va R2=0.569 | tr RMSE=0.634 | va RMSE=0.642 | tr MAE=0.493 | va MAE=0.498 | lr=1.00e-03 | bad_epochs=0
ep=002 tr_loss=0.1946 | va_loss=0.1869 | tr R2=0.622 | va R2=0.589 | tr RMSE=0.614 | va RMSE=0.627 | tr MAE=0.479 | va MAE=0.489 | lr=1.00e-03 | bad_epochs=0
ep=003 tr_loss=0.1746 | va_loss=0.1773 | tr R2=0.653 | va R2=0.607 | tr RMSE=0.589 | va RMSE=0.613 | tr MAE=0.458 | va MAE=0.473 | lr=1.00e-03 | bad_epochs=0
ep=004 tr_loss=0.1701 | va_loss=0.1776 | tr R2=0.654 | va R2=0.609 | tr RMSE=0.588 | va RMSE=0.612 | tr MAE=0.454 | va MAE=0.473 | lr=1.00e-03 | bad_epochs=1
ep=005 tr_loss=0.1677 | va_loss=0.1699 | tr R2=0.673 | va R2=0.628 | tr RMSE=0.572 | va RMSE=0.596 | tr MAE=0.444 | va MAE=0.465 | lr=1.00e-03 | bad_epochs=0
ep=006 tr_loss=0.1652 | va_loss=0.1691 | tr R2=0.673 | va R2=0.633 | tr RMSE=0.572 | va RMSE=0.593 | tr MAE=0.453 | va MAE=0.467 | lr=1.00e-03 | bad_epochs=0
ep=007 tr_loss=0.1638 | va_loss=0.1847 | tr R2=0.642

In [11]:
total_time = np.sum(hist_df["time"].to_numpy())
print(f"Total training time: {total_time} sec.")

Total training time: 4555.065610170364 sec.


In [12]:
m_te = eval_single_metrics_test(model_single, X_test_s, y_test, y_mean, y_std, DEVICE)

single_test_rows = []
single_test_rows.append(["age", m_te["R2"], m_te["RMSE"], m_te["MAE"]])

In [13]:
single_metrics_df = pd.DataFrame(
    single_test_rows,
    columns=["output", "R2", "RMSE", "MAE"]
)
single_metrics_path = f"Results/test_metrics/test_metrics_{NUM_LAYERS}layers.csv"
single_metrics_df.to_csv(single_metrics_path, index=False, float_format="%.6f")

In [14]:
single_metrics_df

,output,R2,RMSE,MAE
0,age,0.69614,7.517903,5.966175
